# SimulationData Pickle Serialization Test

This notebook demonstrates the pickle serialization functionality for saving and loading complete `SimulationData` objects.

**Use Case:** Run simulations in one session, save them, and load in another session for analysis or optimization without re-running.

## Part 1: Run a Simulation and Save to Pickle

In [1]:
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy

In [2]:
# Load parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch"])

# Create simulation
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.FEDBATCH,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    performance_mode=False  # Store detailed data
)

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).


In [3]:
# Run simulation with 3 batches
results = sim.run_simulation(number_of_batches=3)

print(f"Simulation completed:")
print(f"  - Type: {sim.data.simulation_type}")
print(f"  - Batches: {len(sim.data.batches)}")
print(f"  - STY: {sim.data.summary_stats['STY_kg_m3_day']:.2f} kg/m3/day")

run_simulation execution time: 44.358 ms
Simulation completed:
  - Type: Fedbatch
  - Batches: 3
  - STY: 0.09 kg/m3/day


In [4]:
# Save to pickle - preserves EVERYTHING
saved_path = sim.data.save_pickle("demo_fedbatch_results", overwrite=True)

SimulationData object saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\demo_fedbatch_results.pkl


## Part 2: Load the Saved Pickle (Simulates New Session)

Imagine you're in a completely new Python session or notebook. You can load the saved simulation:

In [5]:
from animakinetica.core.simulation_data import SimulationData

# Load the saved simulation data
loaded_data = SimulationData.load_pickle("demo_fedbatch_results")

SimulationData object loaded from: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\demo_fedbatch_results.pkl
  - Simulation type: Fedbatch
  - Number of batches: 3
  - Timestamp: 2026-06-24 11:25:07


## Part 3: Verify Data Integrity

In [6]:
import numpy as np

# Check that loaded data matches original
print("Data Integrity Checks:")
print(f"  ✓ Batch count: {len(loaded_data.batches)}")
print(f"  ✓ Simulation type: {loaded_data.simulation_type}")
print(f"  ✓ STY: {loaded_data.summary_stats['STY_kg_m3_day']:.2f} kg/m3/day")
print(f"  ✓ Parameters preserved: {loaded_data.parameters is not None}")

# Check numpy arrays are intact
batch = loaded_data.batches[0]
print(f"\nBatch 1 Data:")
print(f"  ✓ Timepoints: {len(batch.t_points)} points")
print(f"  ✓ State variables shape: {batch.y_points.shape}")
print(f"  ✓ Growth rate data: {'Available' if batch.mu_list is not None else 'Not stored'}")
print(f"  ✓ Final cell density: {batch.Xv_final:.2e} cells/L")

Data Integrity Checks:
  ✓ Batch count: 3
  ✓ Simulation type: Fedbatch
  ✓ STY: 0.09 kg/m3/day
  ✓ Parameters preserved: True

Batch 1 Data:
  ✓ Timepoints: 43 points
  ✓ State variables shape: (11, 43)
  ✓ Growth rate data: Available
  ✓ Final cell density: 3.12e+09 cells/L


## Part 4: Use Loaded Data for Analysis

All SimulationData methods work on loaded objects!

In [7]:
# Access summary statistics
print("Summary Statistics:")
for key, value in loaded_data.summary_stats.items():
    print(f"  {key}: {value}")

Summary Statistics:
  STY_kg_m3_day: 0.09262994832866991
  medium_use_efficiency_kg_m3: 0.46580729239250557
  protein_use_efficiency_kg_kg: 0.1814646821082273
  nutrient_use_efficiency_kg_kg: 0.09176040810178206
  nutrient_consumption_efficiency_kg_kg: 0.3388358104282197
  batch_time_range_h: 3.6273775094741723e-09


In [8]:
# Visualize loaded data
fig = loaded_data.plot_biomass_metabolites(batch_number=1)
fig.show()

In [9]:
# Export to Excel for sharing (loaded data works identically to original)
loaded_data.save(
    filename="demo_fedbatch_from_pickle",
    add_timestamp=False,
    file_format="excel",
    include_parameters=True,
    include_cultivation_summary=True,
    include_pickle=True,  # Include pickle for full reproducibility
    overwrite=True
)

Simulation data exported to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\demo_fedbatch_from_pickle\demo_fedbatch_from_pickle.xlsx
Parameters saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\demo_fedbatch_from_pickle\demo_fedbatch_from_pickle.parameters.yaml
SimulationData object saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\demo_fedbatch_from_pickle\demo_fedbatch_from_pickle.pkl


WindowsPath('C:/Users/btumulero/OneDrive - Delft University of Technology/Documents/01. Code/animakinetica/export/simulations/demo_fedbatch_from_pickle/demo_fedbatch_from_pickle.xlsx')

In [10]:
# Load the saved simulation data from the exported simulation
loaded_data = SimulationData.load_pickle("demo_fedbatch_from_pickle")

SimulationData object loaded from: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\export\simulations\demo_fedbatch_from_pickle\demo_fedbatch_from_pickle.pkl
  - Simulation type: Fedbatch
  - Number of batches: 3
  - Timestamp: 2026-06-24 11:25:07


## Part 5: Workflow Example - Separate Simulation and Optimization

**Simulation Notebook:**
```python
# Run expensive simulations
sim = CellCultureSimulation(...)
results = sim.run_simulation(number_of_batches=10)

# Save for later
sim.data.save_pickle("baseline_results")
```

**Optimization Notebook (hours/days later):**
```python
# Load baseline results
baseline_data = SimulationData.load_pickle("baseline_results")

# Extract parameters for optimization
baseline_params = baseline_data.parameters

# Create new sim for optimization with baseline comparison
opt_sim = CellCultureSimulation(
    parameters=baseline_params,
    kinetics=Kinetics.SIMPLE,
    operating_mode=OperatingMode.FEDBATCH,
    performance_mode=True  # Fast mode for optimization
)

# Run optimization
from animakinetica.analysis.optimization import OptimizationManager
opt = OptimizationManager(opt_sim, "F_S_optimization")
result = opt.optimize_parameters(objectives=["STY_g_L_day"])

# Compare with baseline
print(f"Baseline STY: {baseline_data.summary_stats['STY_g_L_day']:.2f}")
print(f"Optimized STY: {result['best_objective']:.2f}")
```

## Summary

### New Methods:

1. **`SimulationData.save_pickle(filename, overwrite=False)`**
   - Saves complete simulation state to `.pkl` file
   - Preserves all data: numpy arrays, parameters, metadata
   - Files saved to: `export/simulations/filename.pkl`

2. **`SimulationData.load_pickle(filename)`** (classmethod)
   - Loads saved simulation state
   - Returns fully functional `SimulationData` object
   - Works with relative paths or full paths

### Benefits:
- ✅ **No data loss** - perfect fidelity
- ✅ **Fast** - binary format, efficient storage
- ✅ **Simple** - just save/load, no complex reconstruction
- ✅ **Flexible** - separate simulation from analysis/optimization
- ✅ **Reproducible** - save exact simulation states for papers/reports

### When to Use:
- **Pickle (`.pkl`)**: Internal workflow, full data preservation, Python-to-Python
- **Excel (`.xlsx`)**: Sharing with others, human-readable tables, cross-platform
- **CSV (`.csv`)**: Large datasets, programmatic access, database import